# Decision Tree Classification

This notebook trains and evaluates a Decision Tree to predict the cleaned dataset's existing `burnout_risk_level` labels. It follows the same nine-feature contract, stratified split, and evaluation approach used by the other classifier notebooks. These labels are dataset outputs, not a diagnosis or a validated real-world burnout measure.

## Required imports

We use pandas for tabular data, scikit-learn for splitting, the Decision Tree, and evaluation, matplotlib/seaborn for plots, and joblib for saving and reloading the final model.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.tree import DecisionTreeClassifier

## Project root and dataset path

We look in the current folder and its parent for the project data directory. This supports starting Jupyter from either the project root or the `notebooks` folder.

In [ ]:
dataset_relative_path = Path('data') / 'processed' / 'cognivault_cleaned_dataset.csv'
possible_project_roots = [Path.cwd(), Path.cwd().parent]
project_root = next(
    (path for path in possible_project_roots if (path / dataset_relative_path).exists()),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f'Could not find {dataset_relative_path} from the current folder: {Path.cwd()}'
    )

data_path = project_root / dataset_relative_path
print(f'Project folder: {project_root}')
print(f'Cleaned dataset: {data_path}')

## Load the cleaned dataset

This reads the existing processed CSV. The notebook does not modify or rewrite the dataset.

In [ ]:
df = pd.read_csv(data_path)
print(f'Loaded {len(df)} rows and {len(df.columns)} columns.')
df.head(3)

## Validate the required columns and input features

The classifier must use exactly the nine agreed input columns, in order. The risk score and profile/context fields are not model inputs. These checks catch missing required columns, missing feature values, and labels outside the four expected classes before training.

In [ ]:
features = [
    'study_hours_outside_classes_per_day',
    'academic_tasks_per_week',
    'academic_workload',
    'time_pressure',
    'avg_sleep_hours',
    'wakes_up_tired',
    'physical_activity_days_per_week',
    'difficulty_concentrating',
    'academic_emotional_exhaustion',
]
target = 'burnout_risk_level'
score_target = 'burnout_risk_score'
class_labels = ['Low', 'Moderate', 'High', 'Critical']
profile_context_columns = [
    'institution',
    'degree',
    'year',
    'branch',
    'living_arrangement',
    'student_id',
    'data_source',
]

required_columns = features + [target]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required dataset columns: {missing_columns}')
if len(features) != 9 or len(set(features)) != 9:
    raise ValueError('The classifier must use exactly nine unique input features.')
if target in features or score_target in features:
    raise ValueError('Target columns must not be included in model inputs.')
leaked_profile_columns = [column for column in profile_context_columns if column in features]
if leaked_profile_columns:
    raise ValueError(f'Profile/context columns entered model inputs: {leaked_profile_columns}')
if df[features].isna().any().any():
    raise ValueError('Missing values were found in the selected input features.')

unexpected_labels = sorted(set(df[target].dropna()) - set(class_labels))
if unexpected_labels:
    raise ValueError(f'Unexpected values in {target}: {unexpected_labels}')

print(f'Validated {len(features)} input features in the required order.')
print(f'Prediction target: {target}')
print('Class counts in the cleaned dataset:')
print(df[target].value_counts().reindex(class_labels, fill_value=0).to_string())

## Select the features and target

`X` contains only the nine input features. `y` contains the risk-level labels the tree will learn to predict. The risk score and profile/context fields stay out of `X`.

In [ ]:
X = df.loc[:, features].copy()
y = df[target].copy()

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape: {y.shape}')
print(f'Input columns: {list(X.columns)}')

## Stratified 80/20 train-test split

We set aside 20% of the records for a final test. Stratification keeps the four classes in roughly the same proportions in the training and test sets. `random_state=42` makes the split reproducible. The test set is not used during cross-validation or model fitting.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print(f'Training set: {len(X_train)} rows ({len(X_train) / len(X):.0%})')
print(f'Test set:     {len(X_test)} rows ({len(X_test) / len(X):.0%})')
print('Training class counts:')
print(y_train.value_counts().reindex(class_labels, fill_value=0).to_string())
print('Test class counts:')
print(y_test.value_counts().reindex(class_labels, fill_value=0).to_string())

## How a Decision Tree works

A Decision Tree makes a series of simple questions about feature values—for example, whether a rating is above a chosen value—and follows the resulting branches to choose a class. This notebook starts with a tree that can grow without a fixed depth limit, using `min_samples_split=2` and `min_samples_leaf=1`. We use `class_weight='balanced'` so less frequent labels receive more weight, and `random_state=42` for reproducibility. Trees split on feature thresholds and do not depend on distances between numeric values, so feature scaling is not needed.

## Stratified 5-fold cross-validation on the training data

Five-fold cross-validation divides the training split into five parts. Each time, the model learns from four parts and is checked against the remaining part. Every training row is used for validation once. Stratification preserves class proportions in each fold. We calculate macro F1 and High/Critical recall for each fold and then report their mean and standard deviation. The held-out test data is not used here.

In [ ]:
cross_validator = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_macro_f1_scores = []
cv_high_recall_scores = []
cv_critical_recall_scores = []

for fold_number, (train_indices, validation_indices) in enumerate(
    cross_validator.split(X_train, y_train),
    start=1,
):
    X_fold_train = X_train.iloc[train_indices]
    y_fold_train = y_train.iloc[train_indices]
    X_fold_validation = X_train.iloc[validation_indices]
    y_fold_validation = y_train.iloc[validation_indices]

    fold_model = DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
    )
    fold_model.fit(X_fold_train, y_fold_train)
    fold_predictions = fold_model.predict(X_fold_validation)

    fold_macro_f1 = f1_score(
        y_fold_validation,
        fold_predictions,
        labels=class_labels,
        average='macro',
        zero_division=0,
    )
    fold_high_recall = recall_score(
        y_fold_validation,
        fold_predictions,
        labels=['High'],
        average=None,
        zero_division=0,
    )[0]
    fold_critical_recall = recall_score(
        y_fold_validation,
        fold_predictions,
        labels=['Critical'],
        average=None,
        zero_division=0,
    )[0]

    cv_macro_f1_scores.append(fold_macro_f1)
    cv_high_recall_scores.append(fold_high_recall)
    cv_critical_recall_scores.append(fold_critical_recall)
    print(
        f'Fold {fold_number}: macro F1 = {fold_macro_f1:.4f}, '
        f'High recall = {fold_high_recall:.4f}, '
        f'Critical recall = {fold_critical_recall:.4f}'
    )

print('\\nCross-validation mean (standard deviation):')
print(f'Macro F1:        {pd.Series(cv_macro_f1_scores).mean():.4f} ({pd.Series(cv_macro_f1_scores).std(ddof=0):.4f})')
print(f'High recall:     {pd.Series(cv_high_recall_scores).mean():.4f} ({pd.Series(cv_high_recall_scores).std(ddof=0):.4f})')
print(f'Critical recall: {pd.Series(cv_critical_recall_scores).mean():.4f} ({pd.Series(cv_critical_recall_scores).std(ddof=0):.4f})')

## Train the final Decision Tree

After cross-validation, we create a fresh tree with the same fixed settings and fit it on the complete training split. The held-out test set remains separate until evaluation.

In [ ]:
final_model = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=42,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
)
final_model.fit(X_train, y_train)
print('Final Decision Tree fitted on the complete training split.')

## Evaluate once on the held-out test set

Accuracy is the fraction of all test records classified correctly. Precision tells us how often predictions for a class were correct; recall tells us how many actual records of a class were found. F1 combines precision and recall. Macro F1 averages the four class F1 scores equally. Support is the number of actual test records in each class. We report High and Critical recall separately as well.

In [ ]:
# The test set is used only after cross-validation and final model fitting.
test_predictions = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
test_macro_f1 = f1_score(
    y_test,
    test_predictions,
    labels=class_labels,
    average='macro',
    zero_division=0,
)
test_report = classification_report(
    y_test,
    test_predictions,
    labels=class_labels,
    target_names=class_labels,
    output_dict=True,
    zero_division=0,
)
per_class_metrics = pd.DataFrame(test_report).T.loc[
    class_labels,
    ['precision', 'recall', 'f1-score', 'support'],
]

print(f'Test accuracy: {test_accuracy:.4f}')
print(f'Test macro F1: {test_macro_f1:.4f}')
print('\\nPer-class precision, recall, F1, and support:')
print(per_class_metrics.to_string(float_format=lambda value: f'{value:.4f}'))
print(f"\\nTest High recall:     {test_report['High']['recall']:.4f}")
print(f"Test Critical recall: {test_report['Critical']['recall']:.4f}")

## Confusion matrix

Rows of the confusion matrix are the actual test classes, and columns are the predicted classes. Values on the diagonal are correct predictions; other cells show which labels the model confused.

In [ ]:
confusion_counts = confusion_matrix(
    y_test,
    test_predictions,
    labels=class_labels,
)
confusion_table = pd.DataFrame(
    confusion_counts,
    index=pd.Index(class_labels, name='Actual'),
    columns=pd.Index(class_labels, name='Predicted'),
)
print('Confusion matrix counts (rows = actual, columns = predicted):')
print(confusion_table.to_string())

plt.figure(figsize=(7, 5.5))
sns.heatmap(confusion_table, annot=True, fmt='d', cmap='Blues')
plt.title('Decision Tree: Held-Out Test Confusion Matrix')
plt.xlabel('Predicted risk level')
plt.ylabel('Actual risk level')
plt.tight_layout()
plt.show()

## Feature importance

A Decision Tree can report how much each feature helped reduce class-mixing when the tree made its splits. The chart ranks those model-specific importance scores. A larger score means the fitted tree relied more on that feature for its splits; it does not show that the feature causes burnout.

In [ ]:
feature_importance = pd.Series(
    final_model.feature_importances_,
    index=features,
    name='Importance',
).sort_values()

print('Decision Tree feature importance scores:')
print(feature_importance.sort_values(ascending=False).to_string(float_format=lambda value: f'{value:.4f}'))

plt.figure(figsize=(9, 5))
feature_importance.plot(kind='barh', color='#3498db')
plt.title('Decision Tree Feature Importance')
plt.xlabel('Relative importance')
plt.ylabel('Input feature')
plt.tight_layout()
plt.show()

## Save and reload the Decision Tree

The model uses a distinct filename, `decision_tree_model.joblib`, so it will not replace the Logistic Regression, Random Forest, or SVM artifacts. The save cell checks that the destination does not already exist before writing.

In [ ]:
models_directory = project_root / 'models'
models_directory.mkdir(parents=True, exist_ok=True)
model_path = models_directory / 'decision_tree_model.joblib'

if model_path.exists():
    raise FileExistsError(
        f'Decision Tree model already exists and will not be overwritten: {model_path}'
    )

joblib.dump(final_model, model_path)
print(f'Saved Decision Tree model to: {model_path}')

reloaded_model = joblib.load(model_path)
if list(reloaded_model.feature_names_in_) != features:
    raise ValueError('Reloaded model feature names/order do not match the nine input features.')

sample_features = X_test.iloc[[0]]
sample_prediction = reloaded_model.predict(sample_features)[0]
print(f'Reloaded model prediction for one held-out example: {sample_prediction}')
print('Saved-model reload and prediction check passed.')

## Results and main findings

Run the notebook to see the actual cross-validation and held-out test results, confusion matrix, and feature importance values. Review macro F1 and High/Critical recall alongside accuracy. The results describe prediction of this dataset's existing labels; good scores do not establish clinical validity or reliable real-world burnout detection.

In [ ]:
print('Decision Tree results summary')
print(f'Cross-validation macro F1: {pd.Series(cv_macro_f1_scores).mean():.4f} mean')
print(f'Cross-validation High recall: {pd.Series(cv_high_recall_scores).mean():.4f} mean')
print(f'Cross-validation Critical recall: {pd.Series(cv_critical_recall_scores).mean():.4f} mean')
print(f'Held-out test accuracy: {test_accuracy:.4f}')
print(f'Held-out test macro F1: {test_macro_f1:.4f}')
print(f'Held-out test High recall: {test_report["High"]["recall"]:.4f}')
print(f'Held-out test Critical recall: {test_report["Critical"]["recall"]:.4f}')
print('These metrics measure agreement with the existing dataset labels, not clinical validity.')